# Chapter 10 답안 양식. 분류 분석으로 주문 취소 여부 예측하기

> 이 내용을 `chapter10.ipynb`의 Markdown 셀로 작성합니다. 실제 실행 결과를 근거로 작성하고, 예측 패턴을 원인처럼 단정하지 않습니다.

## 제출 정보
- 이름: 이상재
- GitHub ID: sangjae-lee97
- 작성일: 2026-09-29
- 최종 Notebook URL: https://github.com/sangjae-lee97/kant-axagent-study/blob/main/llm-data-analysis-course/chapter10/chapter10.ipynb

## STEP 0. 제출용 Notebook과 Chapter10 입력 준비

### 설명
이번 실습에서는 data/processed 폴더에 있는 전처리 완료 데이터를 사용합니다.

현재 Notebook 위치가 notebooks/ch10/00.ipynb이므로, 데이터 폴더까지는 ../../data/processed 경로로 접근하면 됩니다.

In [44]:
from pathlib import Path
import pandas as pd

# 데이터 폴더 위치
DATA_DIR = Path("../../data/processed")

# CSV 파일을 DataFrame으로 불러오기
customers = pd.read_csv(DATA_DIR / "customers_clean.csv")
orders = pd.read_csv(DATA_DIR / "orders_clean.csv")
order_items = pd.read_csv(DATA_DIR / "order_items_clean.csv")
products = pd.read_csv(DATA_DIR / "products_clean.csv")

# orders가 제대로 불러와졌는지 확인
print(type(orders))
print(orders.columns)

<class 'pandas.DataFrame'>
Index(['order_id', 'customer_id', 'order_date', 'payment_method',
       'order_status', 'order_month', 'order_dayofweek'],
      dtype='str')


### 결과 해석
각 데이터의 (행 수, 열 수)가 출력되면 정상적으로 불러온 것입니다.
예를 들어 orders: (1000, 8)이면 orders 데이터는 1000행, 8열이라는 뜻입니다.

## STEP 1. Target Contract와 예측 시점 정의

### 설명
이번 실습에서는 주문 상태 중 **`completed`와 `cancelled`만 사용**합니다.

- `completed` → `0`
- `cancelled` → `1`
- `refunded` 등 다른 상태 → 제외

예측 시점은 **주문이 생성된 직후**입니다. 따라서 주문이 취소된 뒤에 생기는 정보는 모델 입력으로 사용하면 안 됩니다.

### 코드 작성

In [45]:
# completed와 cancelled 주문만 선택
model_orders = orders[
    orders["order_status"].isin(["completed", "cancelled"])
].copy()

# Target 생성
model_orders["is_cancelled"] = model_orders["order_status"].map({
    "completed": 0,
    "cancelled": 1
})

# 결과 확인
print(model_orders["order_status"].value_counts())
print()
print(model_orders["is_cancelled"].value_counts())

order_status
completed    184
cancelled     64
Name: count, dtype: int64

is_cancelled
0    184
1     64
Name: count, dtype: int64


### 결과 해석
`order_status`에서는 `completed`, `cancelled`만 남아 있어야 합니다.

`is_cancelled`에서는

```text
0 = 정상 완료 주문
1 = 취소 주문
```

으로 변환된 것을 확인하면 됩니다.

이제부터 모델은 **`is_cancelled`가 0인지 1인지 예측**하게 됩니다.

## STEP 2. 주문 단위 특징과 계산 관계 검증

### 설명
예측 단위는 **주문 1건**이므로 `order_items` 데이터를 `order_id` 기준으로 묶어 주문별 특징을 만듭니다.

먼저 아래 관계가 맞는지 확인합니다.

`line_total = quantity × unit_price`

그다음 주문별로 다음 값을 만듭니다.

- `item_count` : 상품 종류 수
- `total_quantity` : 총 주문 수량
- `order_amount` : 총 주문 금액

### 코드 작성

In [46]:
# line_total 계산이 맞는지 확인
calculated_line_total = order_items["quantity"] * order_items["unit_price"]

line_total_check = (
    calculated_line_total.round(2) == order_items["line_total"].round(2)
)

print("line_total 계산 일치:", line_total_check.all())

# 주문별 특징 생성
order_features = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("product_id", "count"),
        total_quantity=("quantity", "sum"),
        order_amount=("line_total", "sum")
    )
    .reset_index()
)

# 결과 확인
print(order_features.head())
print()
print("주문 수:", len(order_features))

line_total 계산 일치: True
   order_id  item_count  total_quantity  order_amount
0         1           4              14       1436000
1         2           1               4        756000
2         3           3              13       1435000
3         4           2               7        498000
4         5           2               6        941000

주문 수: 300


### 결과 해석

line_total 계산 일치: True이므로 모든 상품 행에서 quantity × unit_price = line_total 관계가 정상입니다.

또한 주문이 총 300건이고, order_features가 주문별로 1행씩 만들어졌습니다. 따라서 주문 단위 특징 생성까지 정상 완료입니다.

## STEP 3. 병합 관계와 Feature Audit 확인

### 설명
방금 만든 주문별 특징을 `orders`와 합칩니다.

이때 중요한 것은 **병합하면서 주문 수가 이상하게 늘어나거나 줄어들지 않는지** 확인하는 것입니다. 이후 모델에 사용하면 안 되는 컬럼도 확인합니다.

### 코드 작성

In [47]:
# 병합 전 주문 수
before_rows = len(model_orders)

# 주문 데이터 + 주문별 특징 병합
model_data = model_orders.merge(
    order_features,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# 병합 후 주문 수
after_rows = len(model_data)

# 주문 특징이 없는 데이터 확인
unmatched_count = model_data["order_amount"].isna().sum()

print("병합 전 주문 수:", before_rows)
print("병합 후 주문 수:", after_rows)
print("미매칭 주문 수:", unmatched_count)

# 모델 입력에 사용하면 안 되는 컬럼
forbidden_features = [
    "order_status",
    "is_cancelled",
    "order_id",
    "customer_id",
    "product_id"
]

# 현재 데이터에 존재하는 금지 컬럼 확인
found_forbidden = [
    col for col in forbidden_features
    if col in model_data.columns
]

print("확인된 금지 컬럼:", found_forbidden)

병합 전 주문 수: 248
병합 후 주문 수: 248
미매칭 주문 수: 0
확인된 금지 컬럼: ['order_status', 'is_cancelled', 'order_id', 'customer_id']


### 결과 해석
병합 전과 후의 주문 수가 같고, 병합 전후 모두 248건이므로 주문 수가 변하지 않았습니다.

미매칭 주문 수: 0이므로 모든 주문에 주문별 특징이 정상적으로 연결되었습니다.

order_status, is_cancelled, order_id, customer_id는 데이터에 존재하지만 모델 입력 Feature에서는 제외해야 합니다.

처음 주문 특징은 300건이었지만 현재 248건인 이유는 STEP 1에서 completed, cancelled만 남기고 다른 주문 상태를 제외했기 때문입니다.

## STEP 4. 클래스 분포와 Dummy Baseline 확인

### 설명
먼저 completed(0)와 cancelled(1)가 각각 얼마나 있는지 확인합니다.

그다음 가장 많은 클래스만 무조건 예측하는 Dummy 모델을 만들어 실제 모델이 최소한 이 기준보다 나은지 비교할 준비를 합니다.

### 코드 작성

In [48]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

# 클래스 개수와 비율 확인
print("클래스 개수")
print(model_data["is_cancelled"].value_counts())

print("\n클래스 비율")
print(model_data["is_cancelled"].value_counts(normalize=True).round(3))

# 임시로 Target만 나누기
X = model_data.drop(columns=["is_cancelled"])
y = model_data["is_cancelled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 가장 많은 클래스만 예측하는 Dummy 모델
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)

print("\nDummy Accuracy:", round(accuracy_score(y_test, dummy_pred), 3))
print("Dummy Cancelled Recall:", round(recall_score(y_test, dummy_pred), 3))

클래스 개수
is_cancelled
0    184
1     64
Name: count, dtype: int64

클래스 비율
is_cancelled
0    0.742
1    0.258
Name: proportion, dtype: float64

Dummy Accuracy: 0.74
Dummy Cancelled Recall: 0.0


![class ratio 비교](images/step02_class_ratio.png)

### 결과 해석
전체 248건 중
- completed(0) : 184건, 약 74.2%
- cancelled(1) : 64건, 약 25.8%

으로 취소 주문이 더 적어서 클래스 불균형이 조금 있습니다.

Dummy 모델의 Accuracy는 0.74로 높아 보이지만, Cancelled Recall = 0.0입니다.

즉, Dummy 모델은 사실상 모든 주문을 completed라고 예측해서 정확도만 높게 나온 것입니다. 취소 주문은 하나도 찾아내지 못합니다.

따라서 이후 실제 모델은 Accuracy만 보지 말고 Precision, Recall, F1도 함께 확인해야 합니다.

## STEP 5. Train / Validation / Test 분리

### 설명

데이터를 세 부분으로 나눕니다.

- Train : 모델 학습
- Validation : 모델과 Threshold 선택
- Test : 마지막 최종 평가

공통 분류 모듈과 동일하게 먼저 Test 20%를 분리하고, 남은 80%에서 Validation 20%를 분리합니다. Test 데이터는 마지막까지 모델 선택에 사용하지 않습니다.

### 코드 작성

In [49]:
from sklearn.model_selection import train_test_split

# Target
y = model_data["is_cancelled"]

# 공통 모듈과 동일하게 Test 20%를 먼저 분리
train_valid_idx, test_idx = train_test_split(
    model_data.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 남은 80%에서 전체의 20%를 Validation으로 분리
train_idx, val_idx = train_test_split(
    train_valid_idx,
    test_size=0.25,
    random_state=42,
    stratify=y.loc[train_valid_idx]
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

print("\nTrain 클래스")
print(y.loc[train_idx].value_counts())

print("\nValidation 클래스")
print(y.loc[val_idx].value_counts())

print("\nTest 클래스")
print(y.loc[test_idx].value_counts())

Train: 148
Validation: 50
Test: 50

Train 클래스
is_cancelled
0    110
1     38
Name: count, dtype: int64

Validation 클래스
is_cancelled
0    37
1    13
Name: count, dtype: int64

Test 클래스
is_cancelled
0    37
1    13
Name: count, dtype: int64


### 결과 해석

데이터가 정상적으로 Train 148 / Validation 50 / Test 50으로 나뉘었습니다.

또한 세 데이터 모두 0과 1이 존재하므로 정상입니다.
- Train → 모델 학습
- Validation → 모델과 Threshold 선택
- Test → 마지막 최종 평가

Test는 마지막 단계까지 건드리지 않습니다.

## STEP 5-1. 모델 입력 컬럼 확인

### 설명

실제 모델에 넣을 Feature를 정하기 전에 현재 데이터에 어떤 컬럼이 있는지 확인합니다.

In [50]:
# 현재 주문 데이터 컬럼 확인
print("model_data 컬럼")
print(model_data.columns.tolist())

print("\ncustomers 컬럼")
print(customers.columns.tolist())

model_data 컬럼
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek', 'is_cancelled', 'item_count', 'total_quantity', 'order_amount']

customers 컬럼
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']


## STEP 5-2. 사용할 Feature 선택

### 설명

공통 분류 모듈과 동일한 Feature Contract를 사용합니다.

숫자형 Feature 6개:
- age
- item_count
- total_quantity
- order_amount
- order_month
- order_dayofweek

범주형 Feature 3개:
- gender
- city
- payment_method

`item_count`, `total_quantity`, `order_amount`는 주문 생성 직후 이미 확인 가능한 주문 정보라고 가정하므로 Feature Leakage가 아닙니다.

반대로 `order_status`, `is_cancelled`, `order_id`, `customer_id` 같은 결과 정보·식별자는 모델 Feature에서 제외합니다. 날짜 품질 문제가 확인된 `days_since_signup`도 사용하지 않습니다.

### 코드 작성

In [51]:
# 재실행해도 고객 컬럼이 중복되지 않도록 정리
model_data = model_data.drop(
    columns=[c for c in ["gender", "age", "city"] if c in model_data.columns],
    errors="ignore"
)

# 고객 정보 추가
model_data = model_data.merge(
    customers[["customer_id", "gender", "age", "city"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# 공통 모듈과 동일하게 날짜 파생 Feature 생성
model_data["order_date"] = pd.to_datetime(model_data["order_date"])
model_data["age"] = pd.to_numeric(model_data["age"], errors="coerce")
model_data["order_month"] = model_data["order_date"].dt.month
model_data["order_dayofweek"] = model_data["order_date"].dt.dayofweek

# 공통 모듈과 동일한 정렬
model_data = (
    model_data
    .sort_values(["order_date", "order_id"])
    .reset_index(drop=True)
)

numeric_features = [
    "age",
    "item_count",
    "total_quantity",
    "order_amount",
    "order_month",
    "order_dayofweek"
]

categorical_features = [
    "gender",
    "city",
    "payment_method"
]

features = numeric_features + categorical_features

X = model_data[features].copy()
y = model_data["is_cancelled"].copy()

print("사용 Feature:")
print(X.columns.tolist())

print("\n데이터 크기:", X.shape)
print("\n결측치:")
print(X.isna().sum())

사용 Feature:
['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'gender', 'city', 'payment_method']

데이터 크기: (248, 9)

결측치:
age                0
item_count         0
total_quantity     0
order_amount       0
order_month        0
order_dayofweek    0
gender             0
city               0
payment_method     0
dtype: int64


### 결과 해석

모델 입력은 총 **9개 Feature**로 구성합니다.

주문 생성 시점에 이미 알 수 있는 주문 수량·금액 정보는 사용할 수 있고, Target·주문 상태·ID·사후 정보는 제외했습니다.

또한 Raw 데이터의 날짜 관계 문제로 신뢰하기 어려운 `days_since_signup`은 모델 Feature에서 제외합니다.

## STEP 6. Pipeline 전처리 확인

### 설명

숫자형과 범주형 Feature는 처리 방법이 다릅니다.

- 숫자형 6개 → 결측치를 중앙값으로 채우고 StandardScaler
- 범주형 3개 → 결측치를 최빈값으로 채우고 OneHotEncoder
- 전처리는 Pipeline 안에서 Train 데이터로만 학습합니다.

### 코드 작성

In [52]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 숫자형 전처리
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# sklearn 버전 호환 OneHotEncoder
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

# 범주형 전처리
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot)
])

# 전처리 합치기
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("숫자형 Feature:", numeric_features)
print("범주형 Feature:", categorical_features)
print("Pipeline 전처리 준비 완료")

숫자형 Feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek']
범주형 Feature: ['gender', 'city', 'payment_method']
Pipeline 전처리 준비 완료


### 결과 해석

공통 분류 모듈과 동일하게 숫자형 Feature **6개**, 범주형 Feature **3개**로 구분했습니다.

`order_month`와 `order_dayofweek`는 날짜에서 만든 숫자형 Feature로 처리합니다.

## STEP 7. Validation에서 후보 모델 비교

### 설명

공통 분류 모듈과 동일한 조건으로 세 모델을 비교합니다.

- Dummy Most Frequent : 기준 모델
- Logistic Regression : `class_weight="balanced"`
- Random Forest : 200 trees, `class_weight="balanced"`

Validation 데이터에서 Accuracy, Precision, Recall, F1을 비교하고, 비베이스라인 모델 중 **F1 → Recall → Precision** 순으로 선택합니다.

### 코드 작성

In [53]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# 공통 모듈과 동일한 60 / 20 / 20 분리
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=0.25,
    random_state=42,
    stratify=y_train_valid
)

models = {
    "Dummy Most Frequent": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1": f1_score(y_val, y_pred, zero_division=0)
    })

validation_results = pd.DataFrame(results)
display(validation_results.round(3))

,Model,Accuracy,Precision,Recall,F1
0,Dummy Most Frequent,0.74,0.000,0.000,0.000
1,Logistic Regression,0.72,0.467,0.538,0.500
2,Random Forest,0.68,0.364,0.308,0.333


![baseline 비교](images/step03_baseline.png)

### 결과 해석

공통 모듈과 동일한 조건으로 비교한 Validation 결과는 다음과 같습니다.

- Dummy Most Frequent → Accuracy **0.74**, Recall **0.000**, F1 **0.000**
- Logistic Regression → Accuracy **0.72**, Precision **0.467**, Recall **0.538**, F1 **0.500**
- Random Forest → Accuracy **0.68**, Precision **0.364**, Recall **0.308**, F1 **0.333**

Dummy는 Accuracy가 높아 보여도 취소 주문을 하나도 찾지 못합니다.

비베이스라인 모델 중 F1이 가장 높은 **Logistic Regression**을 선택합니다.

## STEP 8. Validation에서 Threshold 선택

### 설명

선택한 Logistic Regression의 취소 확률을 이용해 Threshold를 비교합니다.

공통 분류 모듈과 동일하게 **0.20 ~ 0.80을 0.05 간격**으로 비교하고, **F1 → Recall → Precision → 더 낮은 Threshold** 순으로 선택합니다.

### 코드 작성

In [54]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

# 공통 모듈과 동일한 Logistic Regression
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

logistic_pipeline.fit(X_train, y_train)

# Validation 취소 확률
val_prob = logistic_pipeline.predict_proba(X_val)[:, 1]

# 공통 모듈과 동일한 Threshold 범위
thresholds = [round(v, 2) for v in np.arange(0.20, 0.81, 0.05)]

results = []

for threshold in thresholds:
    val_pred = (val_prob >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, val_pred),
        "Precision": precision_score(y_val, val_pred, zero_division=0),
        "Recall": recall_score(y_val, val_pred, zero_division=0),
        "F1": f1_score(y_val, val_pred, zero_division=0)
    })

threshold_results = pd.DataFrame(results)
display(threshold_results.round(3))

# F1 → Recall → Precision → 더 낮은 Threshold
selected_row = threshold_results.sort_values(
    ["F1", "Recall", "Precision", "Threshold"],
    ascending=[False, False, False, True]
).iloc[0]

selected_threshold = float(selected_row["Threshold"])

print("선택된 Threshold:", selected_threshold)

,Threshold,Accuracy,Precision,Recall,F1
0,0.20,0.30,0.271,1.000,0.426
1,0.25,0.34,0.283,1.000,0.441
2,0.30,0.44,0.308,0.923,0.462
3,0.35,0.52,0.333,0.846,0.478
4,0.40,0.62,0.385,0.769,0.513
5,0.45,0.70,0.450,0.692,0.545
6,0.50,0.72,0.467,0.538,0.500
7,0.55,0.76,0.545,0.462,0.500
8,0.60,0.78,0.600,0.462,0.522
9,0.65,0.74,0.500,0.308,0.381


선택된 Threshold: 0.45


![Threshold 비교](images/step05_threshold.png)

### 결과 해석

Validation에서 가장 높은 F1을 만든 Threshold는 **0.45**입니다.

주요 결과는 다음과 같습니다.

- 0.40 → Precision **0.385**, Recall **0.769**, F1 **0.513**
- 0.45 → Precision **0.450**, Recall **0.692**, F1 **0.545**
- 0.50 → Precision **0.467**, Recall **0.538**, F1 **0.500**
- 0.60 → Precision **0.600**, Recall **0.462**, F1 **0.522**

따라서 Validation 기준으로 **Threshold = 0.45**를 고정합니다.

이 값은 9개 Feature와 `class_weight="balanced"`를 사용하는 현재 공통 모델에서 나온 결과입니다.

## STEP 9. 모델과 Threshold를 고정하고 Final Test 평가

### 설명

Validation에서 선택한 값을 고정합니다.

- 선택 모델 → Logistic Regression
- 선택 Threshold → 0.45

이제 Test 결과를 보고 모델이나 Threshold를 다시 바꾸지 않습니다.

### 코드 작성

In [55]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

selected_model_name = "Logistic Regression"

# STEP 8에서 Validation으로 선택한 Threshold 사용
test_prob = logistic_pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= selected_threshold).astype(int)

accuracy = accuracy_score(y_test, test_pred)
precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

print("선택 모델:", selected_model_name)
print("선택 Threshold:", selected_threshold)

print("\nFinal Test 성능")
print("Accuracy :", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall   :", round(recall, 3))
print("F1       :", round(f1, 3))

print("\nConfusion Matrix")
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

선택 모델: Logistic Regression
선택 Threshold: 0.45

Final Test 성능
Accuracy : 0.6
Precision: 0.316
Recall   : 0.462
F1       : 0.375

Confusion Matrix
TN: 24
FP: 13
FN: 7
TP: 6


### 결과 해석

Final Test에서 `Logistic Regression + Threshold 0.45`의 성능은 다음과 같습니다.

- Accuracy = **0.600**
- Precision = **0.316**
- Recall = **0.462**
- F1 = **0.375**

Confusion Matrix는 다음과 같습니다.

- TN = **24**
- FP = **13**
- FN = **7**
- TP = **6**

즉 실제 취소 주문 13건 중 **6건을 찾아냈고 7건을 놓쳤습니다.**

반대로 정상 주문 37건 중 **13건을 취소 위험으로 잘못 판단**했습니다.

Test 결과는 최종 평가이므로 이 결과를 보고 Threshold를 다시 조정하지 않습니다.

## STEP 10. Confusion Matrix와 FP/FN 해석

### 설명

이번 단계에서는 모델이 틀린 경우를 두 종류의 실수로 나눠서 봅니다.
- FP: 실제로는 정상 주문인데 취소 위험이라고 잘못 판단
- FN: 실제로는 취소 주문인데 정상이라고 놓침

이번 실습에서는 단순히 숫자만 보는 게 아니라, FP와 FN이 실제 업무에서 어떤 비용을 만드는지 생각해야 합니다.

### 코드 작성

In [56]:
# Confusion Matrix 값 정리
confusion_result = pd.DataFrame({
    "구분": ["TN", "FP", "FN", "TP"],
    "건수": [tn, fp, fn, tp],
    "의미": [
        "정상 주문을 정상으로 맞춤",
        "정상 주문을 취소 위험으로 잘못 판단",
        "실제 취소 주문을 정상으로 놓침",
        "실제 취소 주문을 취소 위험으로 맞춤"
    ]
})

display(confusion_result)

# 오류 비율 확인
print("FP 비율:", round(fp / (tn + fp), 3))
print("FN 비율:", round(fn / (tp + fn), 3))

,구분,건수,의미
0,TN,24,정상 주문을 정상으로 맞춤
1,FP,13,정상 주문을 취소 위험으로 잘못 판단
2,FN,7,실제 취소 주문을 정상으로 놓침
3,TP,6,실제 취소 주문을 취소 위험으로 맞춤


FP 비율: 0.351
FN 비율: 0.538


![Confusion Matrix](images/step06_confusion_matrix.png)

### 결과 해석

이번 Test 데이터 50건에서:

- TN **24건** → 정상 주문을 정상으로 맞춤
- FP **13건** → 정상 주문을 취소 위험으로 잘못 판단
- FN **7건** → 실제 취소 주문을 정상으로 놓침
- TP **6건** → 실제 취소 주문을 취소 위험으로 맞춤

FP 비율은 `13 / 37 ≈ 0.351`이므로 정상 주문의 약 **35.1%를 잘못 경고**했습니다.

FN 비율은 `7 / 13 ≈ 0.538`이므로 실제 취소 주문의 약 **53.8%를 놓쳤습니다.**

따라서 현재 모델은 잘못된 경고와 놓친 취소가 모두 존재합니다. 실제 운영에서는 FP와 FN 중 어떤 비용이 더 큰지 업무 기준을 추가로 정해야 합니다.

## STEP 11. Internal 결과와 Public 결과 구분

### 설명

모델 오류를 분석할 때는 order_id, customer_id 같은 식별자가 내부적으로 필요할 수 있습니다.

하지만 외부에 공개하는 예측 결과에서는 이런 식별자를 제거해야 합니다.

따라서 이번 단계에서는 결과를 두 가지로 나눕니다.

- Internal → 분석용이므로 식별자 포함
- Public → 공개용이므로 식별자 제거

### 코드 작성

In [57]:
import pandas as pd

# 내부 분석용 결과
internal_predictions = pd.DataFrame({
    "source_index": X_test.index,
    "order_id": model_data.loc[X_test.index, "order_id"].values,
    "customer_id": model_data.loc[X_test.index, "customer_id"].values,
    "actual_is_cancelled": y_test.values,
    "predicted_is_cancelled": test_pred,
    "cancel_probability": test_prob
})

print("Internal 결과")
display(internal_predictions.head())

# 공개용 결과
public_predictions = pd.DataFrame({
    "record_id": [f"test_{i:04d}" for i in range(1, len(y_test) + 1)],
    "actual_is_cancelled": y_test.values,
    "predicted_is_cancelled": test_pred,
    "cancel_probability": test_prob,
    "model": selected_model_name,
    "threshold": selected_threshold
})

print("\nPublic 결과")
display(public_predictions.head())

# 공개 금지 컬럼 확인
forbidden_columns = [
    "source_index",
    "order_id",
    "customer_id",
    "product_id"
]

found_forbidden = [
    col for col in forbidden_columns
    if col in public_predictions.columns
]

print("\nPublic 결과에 포함된 금지 컬럼:", found_forbidden)

Internal 결과


,source_index,order_id,customer_id,actual_is_cancelled,predicted_is_cancelled,cancel_probability
0,28,267,5,0,1,0.683535
1,146,9,145,0,0,0.409431
2,211,28,31,0,1,0.512531
3,125,52,141,0,1,0.602233
4,77,77,26,0,1,0.746377



Public 결과


,record_id,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,0,1,0.683535,Logistic Regression,0.45
1,test_0002,0,0,0.409431,Logistic Regression,0.45
2,test_0003,0,1,0.512531,Logistic Regression,0.45
3,test_0004,0,1,0.602233,Logistic Regression,0.45
4,test_0005,0,1,0.746377,Logistic Regression,0.45



Public 결과에 포함된 금지 컬럼: []


### 결과 해석

Internal 결과에는 오류 분석에 필요한 `source_index`, `order_id`, `customer_id`가 포함됩니다.

Public 결과에는 식별자를 제거하고 다음 정보만 남깁니다.

- record_id
- 실제값
- 예측값
- 취소 확률
- 모델명
- Threshold

`Public 결과에 포함된 금지 컬럼: []`이면 공개용 결과에서 식별자가 정상적으로 제거된 것입니다.

## STEP 12. 공통 분류 모델 실행 및 Validation Evidence 확인

### 설명

마지막으로 공통 분류 모듈을 실행하여 Notebook에서 사용한 Feature, 모델 선택, Threshold 선택, Test 평가와 자동 검증 결과가 일치하는지 확인합니다.

Raw 데이터에서 `signup_date > order_date`인 주문 45건이 확인되어 원본 데이터는 수정하지 않고 보존했습니다. 해당 날짜 관계에서 만들어지는 `days_since_signup`은 모델 Feature에서 제외하고, 45건은 데이터 품질 문제로 기록했습니다.

### 코드 작성

In [58]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path("../..").resolve()
script_path = PROJECT_ROOT / "scripts" / "run_classification_analysis.py"

result = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

Chapter10 분류 분석 완료

[모델링 데이터]
(248, 16)

[타깃 분포]
 is_cancelled class_label  count  ratio
            0   completed    184 0.7419
            1   cancelled     64 0.2581

[Feature Audit]
         column  selected            role                 reason
            age      True allowed_feature 교육용 예측 시점에 사용 가능하다고 가정
     item_count      True allowed_feature 교육용 예측 시점에 사용 가능하다고 가정
 total_quantity      True allowed_feature 교육용 예측 시점에 사용 가능하다고 가정
   order_amount      True allowed_feature 교육용 예측 시점에 사용 가능하다고 가정
    order_month      True allowed_feature 교육용 예측 시점에 사용 가능하다고 가정
order_dayofweek      True allowed_feature 교육용 예측 시점에 사용 가능하다고 가정
         gender      True allowed_feature 교육용 예측 시점에 사용 가능하다고 가정
           city      True allowed_feature 교육용 예측 시점에 사용 가능하다고 가정
 payment_method      True allowed_feature 교육용 예측 시점에 사용 가능하다고 가정
  cancel_reason     False       forbidden       취소 이후 생성되는 사후 정보
   cancelled_at     False       forbidden       취소 이후 생성되는 사후 정보
    customer_id     False       fo

### 결과 해석

공통 분류 분석이 정상적으로 완료되었으며 자동 Validation Evidence가 모두 **PASS**했습니다.

공통 모듈과 Notebook 모두 동일하게:

- Feature **9개**
- Logistic Regression (`class_weight="balanced"`)
- Validation 선택 Threshold **0.45**
- Final Test Accuracy **0.600**
- Precision **0.316**
- Recall **0.462**
- F1 **0.375**

를 사용합니다.

또한 공개 Prediction에는 `order_id`, `customer_id` 등의 식별자가 포함되지 않아 Privacy 검증도 통과했습니다.

## STEP 13. 전체 스크립트 재실행

### 결과 해석

STEP 13 전체 재실행이 정상적으로 완료되었습니다. `prepare_ch10_data.py`에서 고객·주문·상품 간 관계 검증이 모두 `invalid_count = 0`으로 통과했고, 전처리된 CSV 파일들도 정상적으로 다시 저장되었습니다. 이후 `run_classification_analysis.py`에서도 Notebook과 동일하게 총 248건의 모델링 데이터와 9개 Feature를 사용했으며, Train / Validation / Test는 148 / 50 / 50으로 분리되었습니다. Validation에서 Logistic Regression이 선택되었고 Threshold는 0.45로 결정되었습니다. Final Test 결과는 Accuracy 0.600, Precision 0.316, Recall 0.462, F1 0.375였으며, Confusion Matrix는 TN 24, FP 13, FN 7, TP 6으로 나타났습니다. 또한 Target Contract, Feature Leakage, 병합 관계, 데이터 분할, 모델 선택, Threshold 선택, Public Prediction Privacy 검증이 모두 PASS했습니다. 따라서 Notebook과 Script가 동일한 분석 조건으로 정상 실행되었음을 확인할 수 있습니다.

## STEP 14. LLM 검토와 최종 사용 판단

### 설명

이번 단계에서는 새로운 모델을 만드는 것이 아니라, 지금까지의 분석 계약과 결과를 바탕으로 모델을 실제로 어느 수준까지 사용할 수 있는지 검토합니다.

LLM 검토에서는 원본 고객 데이터를 전달할 필요 없이 Target 정의, 예측 시점, 허용·금지 Feature, Train/Validation/Test 역할, 모델·Threshold 선택 규칙, Final Test 보호 규칙, 평가 지표, Privacy-safe Output 규칙을 확인하면 됩니다. 

또한 Test 데이터로 모델이나 Threshold를 다시 선택하거나, order_status 같은 누출 Feature를 사용하는 제안은 받아들이면 안 됩니다.

### 결과 해석

이번 분류 모델은 completed=0, cancelled=1을 Target으로 사용하고 refunded 등 다른 상태는 제외했습니다. 전체 248건 중 completed는 184건(74.2%), cancelled는 64건(25.8%)으로 클래스 불균형이 존재합니다. Dummy 모델은 Accuracy 0.74였지만 Recall과 F1이 모두 0이어서 취소 주문을 전혀 탐지하지 못했습니다. 이에 비해 Validation에서 선택된 Logistic Regression은 취소 주문을 실제로 탐지할 수 있었으며, Validation에서 F1이 가장 높았던 Threshold 0.45를 최종 기준으로 고정했습니다.

Final Test 결과는 Accuracy 0.600, Precision 0.316, Recall 0.462, F1 0.375였으며, TN 24, FP 13, FN 7, TP 6으로 나타났습니다. 즉 실제 취소 주문 13건 중 6건을 탐지했지만 7건을 놓쳤고, 정상 주문 37건 중 13건을 취소 위험으로 잘못 판단했습니다. 따라서 FN으로 인한 실제 취소 누락 비용과 FP로 인한 불필요한 대응 비용을 모두 고려해야 합니다.

또한 현재 평가는 Random Split을 사용한 교육용 분석이므로 실제 운영 환경의 미래 데이터에서도 같은 성능이 유지된다고 보장할 수 없습니다. 운영 적용 전에는 새로운 기간의 데이터에 대한 Out-of-Time 검증과 추가적인 성능 확인이 필요합니다. 공개 Prediction에서는 order_id, customer_id 등의 원본 식별자를 제거하여 Privacy-safe Output 조건을 충족했습니다.

따라서 현재 모델은 바로 운영에 적용하기보다는 **추가 검증 후 사용 가능**으로 판단합니다. 현재 단계에서는 분석 및 참고용으로 활용할 수 있지만, 실제 자동 의사결정에 사용하기 전에는 새로운 데이터에 대한 성능 검증과 FP/FN 비용 기준을 추가로 정의해야 합니다.

### step15. streamlit으로 실행하기

![streamlit 화면](./images/step07_streamlit.png)

### 결과 해석

Streamlit에서 9개 Feature를 직접 입력하여 모델의 예측 결과를 확인했습니다. 

입력값은 age=20, item_count=1, total_quantity=1, order_amount=804000, order_month=12, order_dayofweek=0(월요일), gender=M, city=대전, payment_method=card였습니다. 

Logistic Regression 모델은 이 주문의 취소 확률을 **74.2%**로 예측했으며, Validation에서 고정한 Threshold **45%**보다 높기 때문에 최종적으로 **취소 위험 주문(1)**으로 분류했습니다. 

이를 통해 사용자가 Feature 값을 직접 변경하면서 취소 확률과 최종 예측 결과가 어떻게 달라지는지 Streamlit 화면에서 확인할 수 있습니다. 

다만 이 확률은 특정 Feature가 취소의 원인이라는 의미가 아니라, 학습 데이터에서 해당 Feature 조합을 바탕으로 모델이 추정한 취소 가능성입니다.